In [13]:

# Task 1: Dataset Collection
import pandas as pd
movies_df = pd.read_csv('/content/movies.csv')

ratings_df = pd.read_csv(
    '/content/ratings.csv',
    usecols=['userId', 'movieId', 'rating']
)
print("Movies Dataset:")
display(movies_df.head())

print("Ratings Dataset:")
display(ratings_df.head())

print("Movies Shape:", movies_df.shape)
print("Ratings Shape:", ratings_df.shape)

print("Number of Users:", ratings_df['userId'].nunique())
print("Number of Movies:", movies_df['movieId'].nunique())

print("\nTask 1 Completed Successfully!")


Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


Movies Shape: (87585, 3)
Ratings Shape: (32000204, 3)
Number of Users: 200948
Number of Movies: 87585

Task 1 Completed Successfully!


In [14]:

# Task 2: Content Feature Extraction

from sklearn.feature_extraction.text import TfidfVectorizer

# 2. Prepare Movie Genres
movies_df['content_text'] = movies_df['genres'].fillna('').str.replace('|', ' ', regex=False).str.replace('Sci-Fi', 'SciFi', regex=False)

# 3. Remove Movies Without Genres
movies_df['content_text'] = movies_df['content_text'].str.replace('(no genres listed)', '', regex=False)

# 4. Apply TF-IDF
tfidf = TfidfVectorizer(stop_words='english')

# 5. Generate Content Feature Vectors
tfidf_matrix = tfidf.fit_transform(movies_df['content_text'])

# 6. Get Feature Names
feature_names = tfidf.get_feature_names_out()

# 7. Display Results
print("Content Text:")
display(movies_df[['movieId', 'title', 'content_text']].head())

print("\nTF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

print("\nNumber of Content Features:")
print(len(feature_names))

print("\nContent Feature Names:")
print(feature_names)

# 8. Display Sample Feature Vectors
print("\nSample TF-IDF Vectors:")
display(pd.DataFrame(tfidf_matrix[:5].toarray(), columns=feature_names).round(3))

print("\nTask 2 Completed Successfully!")


Content Text:


,movieId,title,content_text
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy



TF-IDF Matrix Shape:
(87585, 20)

Number of Content Features:
20

Content Feature Names:
['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'film' 'horror' 'imax' 'musical'
 'mystery' 'noir' 'romance' 'scifi' 'thriller' 'war' 'western']

Sample TF-IDF Vectors:


,action,adventure,animation,children,comedy,crime,documentary,drama,fantasy,film,horror,imax,musical,mystery,noir,romance,scifi,thriller,war,western
0,0.0,0.459,0.478,0.481,0.283,0.0,0.0,0.000,0.500,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0
1,0.0,0.552,0.000,0.578,0.000,0.0,0.0,0.000,0.601,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0
2,0.0,0.000,0.000,0.000,0.597,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.802,0.0,0.0,0.0,0.0
3,0.0,0.000,0.000,0.000,0.535,0.0,0.0,0.445,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.718,0.0,0.0,0.0,0.0
4,0.0,0.000,0.000,0.000,1.000,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0



Task 2 Completed Successfully!


In [15]:

# Task 3: Collaborative Feature Extraction

# 1. Install and Import Libraries
!pip install scikit-surprise -q

import numpy as np
from scipy.sparse import csr_matrix
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

# 2. Select a Sample from Ratings
ratings_sample = ratings_df.sample(
    n=1000000,
    random_state=42
)

# 3. Construct User-Item Interaction Matrix
user_index, user_ids = pd.factorize(ratings_sample['userId'])

movie_index, movie_ids = pd.factorize(ratings_sample['movieId'])

user_item_matrix = csr_matrix(
    (ratings_sample['rating'].values, (user_index, movie_index)),
    shape=(len(user_ids), len(movie_ids))
)

# 4. Display User-Item Matrix
print("User-Item Matrix Shape:", user_item_matrix.shape)

display(pd.DataFrame(
    user_item_matrix[:5, :5].toarray(),
    index=user_ids[:5],
    columns=movie_ids[:5]
))

# 5. Prepare Data for SVD
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_sample[['userId', 'movieId', 'rating']],
    reader
)

# 6. Split Data into Training and Testing
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

# 7. Apply SVD
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.05,
    random_state=42
)

svd_model.fit(trainset)

# 8. Evaluate SVD Model
predictions = svd_model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

# 9. Extract User Latent Features
user_factors = pd.DataFrame(
    svd_model.pu,
    index=[trainset.to_raw_uid(i) for i in range(trainset.n_users)],
    columns=[f'Factor_{i+1}' for i in range(svd_model.n_factors)]
)

# 10. Extract Item Latent Features
item_factors = pd.DataFrame(
    svd_model.qi,
    index=[trainset.to_raw_iid(i) for i in range(trainset.n_items)],
    columns=[f'Factor_{i+1}' for i in range(svd_model.n_factors)]
)

# 11. Display Results
print("\nUser Factors Shape:", user_factors.shape)

display(user_factors.iloc[:5, :5])

print("\nItem Factors Shape:", item_factors.shape)

display(item_factors.iloc[:5, :5])

print("\nTask 3 Completed Successfully!")


User-Item Matrix Shape: (167266, 27726)


,781,574,1088,2706,275079
66954,5.0,0.0,0.0,0.0,0.0
9877,0.0,4.0,0.0,0.0,0.0
38348,0.0,0.0,2.0,0.0,0.0
101952,0.0,0.0,0.0,1.0,0.0
140400,0.0,0.0,0.0,0.0,3.5


RMSE: 0.9062
MAE:  0.6954

User Factors Shape: (157746, 50)


,Factor_1,Factor_2,Factor_3,Factor_4,Factor_5
58628,0.050783,-0.011917,0.063127,0.150734,-0.014831
115021,0.024959,-0.055075,-0.061844,0.048541,0.124808
133641,-0.134097,-0.059845,-0.036989,-0.069064,0.003198
20296,0.017259,0.052225,-0.074735,0.011955,0.028699
50368,0.031856,0.056382,0.074589,0.109814,-0.138460



Item Factors Shape: (25506, 50)


,Factor_1,Factor_2,Factor_3,Factor_4,Factor_5
258451,0.061630,0.005260,-0.012280,0.077524,0.156453
122882,0.017813,0.020037,-0.131725,0.059957,0.007480
2141,-0.030353,-0.191472,0.047626,0.048256,-0.129510
168250,0.046562,-0.109019,-0.006019,0.076801,-0.036477
1215,-0.170797,0.124136,0.090904,0.074494,0.184673



Task 3 Completed Successfully!


In [16]:

# Task 4: Feature Combination

# 1. Import Libraries
from sklearn.preprocessing import normalize
from scipy.sparse import hstack, csr_matrix

# 2. Match Movies Using Movie ID
item_factors.index = item_factors.index.astype(int)

common_movies = movies_df[
    movies_df['movieId'].isin(item_factors.index)
]

common_movie_ids = common_movies['movieId'].values

# 3. Extract Content and Collaborative Features
C = tfidf_matrix[common_movies.index]

F = item_factors.loc[common_movie_ids].values

# 4. Normalize Both Feature Matrices
C_norm = normalize(C, norm='l2')
F_norm = normalize(F, norm='l2')

# 5. Set Weighting Parameters
alpha = 0.5
beta = 0.5

# 6. Combine Features Using Concatenation
hybrid_matrix = hstack([
    alpha * C_norm,
    beta * csr_matrix(F_norm)
]).tocsr()

# 7. Save Hybrid Movie IDs
hybrid_movie_ids = common_movie_ids

# 8. Display Results
print("Content Features Shape:", C.shape)

print("Collaborative Features Shape:", F.shape)

print("Number of Common Movies:", len(common_movie_ids))

print("\nHybrid Matrix Shape:", hybrid_matrix.shape)

print("\nHybrid Feature Vectors:")

hybrid_columns = (
    [f'Content_{i+1}' for i in range(C.shape[1])] +
    [f'Factor_{i+1}' for i in range(F.shape[1])]
)

display(pd.DataFrame(
    hybrid_matrix[:5].toarray(),
    index=hybrid_movie_ids[:5],
    columns=hybrid_columns
).round(3))

print("\nTask 4 Completed Successfully!")


Content Features Shape: (25506, 20)
Collaborative Features Shape: (25506, 50)
Number of Common Movies: 25506

Hybrid Matrix Shape: (25506, 70)

Hybrid Feature Vectors:


,Content_1,Content_2,Content_3,Content_4,Content_5,Content_6,Content_7,Content_8,Content_9,Content_10,...,Factor_41,Factor_42,Factor_43,Factor_44,Factor_45,Factor_46,Factor_47,Factor_48,Factor_49,Factor_50
1,0.0,0.229,0.239,0.240,0.141,0.0,0.0,0.000,0.250,0.0,...,-0.126,-0.070,0.012,0.104,-0.008,-0.034,-0.121,0.021,0.027,0.069
2,0.0,0.276,0.000,0.289,0.000,0.0,0.0,0.000,0.301,0.0,...,-0.048,-0.022,0.043,-0.011,0.032,0.150,-0.099,0.047,0.069,0.131
3,0.0,0.000,0.000,0.000,0.298,0.0,0.0,0.000,0.000,0.0,...,-0.078,-0.107,-0.034,-0.126,0.033,-0.040,0.118,-0.036,-0.074,0.030
4,0.0,0.000,0.000,0.000,0.267,0.0,0.0,0.223,0.000,0.0,...,0.067,-0.077,-0.046,-0.042,0.078,-0.025,-0.055,-0.068,-0.144,-0.020
5,0.0,0.000,0.000,0.000,0.500,0.0,0.0,0.000,0.000,0.0,...,-0.065,-0.069,-0.061,0.029,0.009,0.028,0.068,0.024,0.073,0.019



Task 4 Completed Successfully!


In [17]:

# Task 5: Recommendation Generation

# 1. Import Libraries
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 2. Create Movie ID Mapping
hybrid_movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(hybrid_movie_ids)
}

# 3. Prepare User Ratings
profile_ratings = ratings_sample[
    ratings_sample['movieId'].isin(hybrid_movie_ids)
].copy()

# 4. Select Three Different Users
user_counts = profile_ratings.groupby('userId')['movieId'].nunique()

selected_users = user_counts.sort_values(
    ascending=False
).head(3).index.tolist()

# 5. Get All Rated Movies for Selected Users
selected_user_history = ratings_df[
    ratings_df['userId'].isin(selected_users)
]

# 6. Recommendation Function
def recommend_movies(user_id, top_n=10):

    # Get User Ratings
    user_ratings = profile_ratings[
        profile_ratings['userId'] == user_id
    ]

    # Get Hybrid Vectors of Rated Movies
    rated_indices = user_ratings['movieId'].map(
        hybrid_movie_to_index
    ).values

    rated_vectors = hybrid_matrix[rated_indices]

    # Get User Ratings as Weights
    ratings = user_ratings['rating'].values.reshape(-1, 1)

    # Build User Profile
    user_profile = np.asarray(
        rated_vectors.multiply(ratings).sum(axis=0) / ratings.sum()
    ).reshape(1, -1)

    # Calculate Cosine Similarity
    similarity_scores = cosine_similarity(
        user_profile,
        hybrid_matrix
    ).flatten()

    # Create Recommendations DataFrame
    recommendations = common_movies[
        ['movieId', 'title', 'genres']
    ].copy().reset_index(drop=True)

    # Add Similarity Scores
    recommendations['similarity'] = similarity_scores

    # Get All Movies Already Rated by User
    rated_movie_ids = set(
        selected_user_history.loc[
            selected_user_history['userId'] == user_id,
            'movieId'
        ]
    )

    # Exclude Rated Movies
    recommendations = recommendations[
        ~recommendations['movieId'].isin(rated_movie_ids)
    ]

    # Sort Recommendations
    recommendations = recommendations.sort_values(
        by='similarity',
        ascending=False
    )

    # Return Top-N Recommendations
    return recommendations.head(top_n).reset_index(drop=True)


# 7. Test the System on Three Users
for user_id in selected_users:

    print(f"\nRecommendations for User {user_id}")

    user_ratings = profile_ratings[
        profile_ratings['userId'] == user_id
    ]

    print("Number of Rated Movies:", len(user_ratings))

    recommendations = recommend_movies(
        user_id,
        top_n=10
    )

    display(
        recommendations.round({
            'similarity': 4
        })
    )

print("\nTask 5 Completed Successfully!")



Recommendations for User 175325
Number of Rated Movies: 957


,movieId,title,genres,similarity
0,258677,Swan Song (2021),Comedy|Drama,0.5967
1,157681,Gotta Kick It Up (2002),Comedy|Drama,0.5955
2,75,Big Bully (1996),Comedy|Drama,0.5932
3,70968,"Dog Year, A (2009)",Comedy|Drama,0.5920
4,52326,Double Harness (1933),Comedy|Drama,0.5911
5,213904,Life After Death (2020),Comedy|Drama,0.5895
6,90649,Mr. Nice (2010),Comedy|Drama,0.5888
7,218733,Feel the Beat (2020),Comedy|Drama,0.5885
8,3410,Soft Fruit (1999),Comedy|Drama,0.5885
9,263361,Het Sacrament (1990),Comedy|Drama,0.5882



Recommendations for User 17035
Number of Rated Movies: 294


,movieId,title,genres,similarity
0,8330,Our Man in Havana (1959),Comedy|Drama|Thriller,0.6055
1,104451,Dealing: Or the Berkeley-to-Boston Forty-Brick...,Comedy|Drama|Thriller,0.5970
2,124509,The Job (2009),Comedy|Drama|Thriller,0.5938
3,109,Headless Body in Topless Bar (1995),Comedy|Drama|Thriller,0.5906
4,82097,Karthik Calling Karthik (2010),Comedy|Drama|Thriller,0.5893
5,199281,Russian Psycho (2019),Comedy|Drama|Thriller,0.5885
6,193327,Best F(r)iends: Volume One (2018),Comedy|Drama|Thriller,0.5880
7,65111,Two Days (2003),Comedy|Drama|Thriller,0.5832
8,144254,Bad Day to Go Fishing (2009),Comedy|Drama|Thriller,0.5803
9,158565,Rich Kids (1979),Comedy|Drama|Thriller,0.5769



Recommendations for User 123465
Number of Rated Movies: 272


,movieId,title,genres,similarity
0,186749,Hichki (2018),Comedy|Drama,0.5490
1,144138,Billy Elliot: The Musical (2014),Comedy|Drama,0.5454
2,113315,Zero Motivation (Efes beyahasei enosh) (2014),Comedy|Drama,0.5400
3,168760,Cosy Dens (1999),Comedy|Drama,0.5394
4,188311,Ideal Home (2018),Comedy|Drama,0.5391
5,253900,The Space Between (2021),Comedy|Drama,0.5370
6,206451,Bad Education (2019),Comedy|Drama,0.5368
7,57980,"Art of Negative Thinking, The (Kunsten å tenke...",Comedy|Drama,0.5363
8,107141,Saving Mr. Banks (2013),Comedy|Drama,0.5354
9,175097,All the Wrong Reasons (2013),Comedy|Drama,0.5350



Task 5 Completed Successfully!
